# 03 - Kafka Producer
Simulateur de flux : lit le CSV et envoie chaque trajet dans Kafka

In [1]:
# Installer kafka-python
import subprocess
subprocess.check_call(['pip', 'install', 'kafka-python'])
print('kafka-python installe')

kafka-python installe


In [2]:
from kafka import KafkaProducer
import json, time, csv

KAFKA_BROKER = 'kafka:29092'
TOPIC = 'nyc-taxi-trips'

# Connexion au broker Kafka
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print(f'Producer connecte a {KAFKA_BROKER}')
print(f'Topic : {TOPIC}')

Producer connecte a kafka:29092
Topic : nyc-taxi-trips


In [3]:
# Envoyer les trajets dans Kafka (1 message = 1 trajet)
CSV_PATH = '/home/jovyan/data/fact_trips_sample.csv'
MAX_MESSAGES = 25000   # Nombre de messages a envoyer
BATCH_SIZE = 500       # Afficher un log tous les X messages
DELAY = 0.001          # Delai entre chaque message (simule le temps reel)

count = 0
with open(CSV_PATH, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convertir les types
        message = {
            'date_id': float(row['data_id']) if row['data_id'] else None,
            'pickup_location_id': int(row['localizacao_partida_id']),
            'dropoff_location_id': int(row['localizacao_chegada_id']),
            'weather_id': float(row['clima_id']) if row['clima_id'] else None,
            'trip_distance': float(row['distancia_viagem']),
            'total_amount': float(row['valor_total'])
        }
        producer.send(TOPIC, value=message)
        count += 1

        if count % BATCH_SIZE == 0:
            print(f'  {count} messages envoyes...')

        if count >= MAX_MESSAGES:
            break

        time.sleep(DELAY)

producer.flush()
print(f'\nTermine : {count} messages envoyes dans le topic "{TOPIC}"')

  500 messages envoyes...
  1000 messages envoyes...
  1500 messages envoyes...
  2000 messages envoyes...
  2500 messages envoyes...
  3000 messages envoyes...
  3500 messages envoyes...
  4000 messages envoyes...
  4500 messages envoyes...
  5000 messages envoyes...
  5500 messages envoyes...
  6000 messages envoyes...
  6500 messages envoyes...
  7000 messages envoyes...
  7500 messages envoyes...
  8000 messages envoyes...
  8500 messages envoyes...
  9000 messages envoyes...
  9500 messages envoyes...
  10000 messages envoyes...
  10500 messages envoyes...
  11000 messages envoyes...
  11500 messages envoyes...
  12000 messages envoyes...
  12500 messages envoyes...
  13000 messages envoyes...
  13500 messages envoyes...
  14000 messages envoyes...
  14500 messages envoyes...
  15000 messages envoyes...
  15500 messages envoyes...
  16000 messages envoyes...
  16500 messages envoyes...
  17000 messages envoyes...
  17500 messages envoyes...
  18000 messages envoyes...
  18500 mess

In [4]:
# Verifier que les messages sont bien dans Kafka
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BROKER,
    auto_offset_reset='earliest',
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
    consumer_timeout_ms=5000
)

sample = []
for msg in consumer:
    sample.append(msg.value)
    if len(sample) >= 3:
        break

consumer.close()
print(f'Apercu des messages dans "{TOPIC}" :')
for s in sample:
    print(f'  {s}')

Apercu des messages dans "nyc-taxi-trips" :
  {'date_id': 327.0, 'pickup_location_id': 168, 'dropoff_location_id': 159, 'weather_id': None, 'trip_distance': 1.14, 'total_amount': 16.8}
  {'date_id': 257.0, 'pickup_location_id': 79, 'dropoff_location_id': 41, 'weather_id': None, 'trip_distance': 8.46, 'total_amount': 52.44}
  {'date_id': 268.0, 'pickup_location_id': 70, 'dropoff_location_id': 79, 'weather_id': None, 'trip_distance': 9.7, 'total_amount': 69.7}
